# NUTDTS 816 Time Series Analysis
## L02 Visualisation and autocorrelation

Lab notebook for Chapter 1 of the lecture notes. Run the setup cell first. Every code cell reproduces an example from the notes; the exercises at the end are from the chapter's self-check list.

**Instructor:** Dr Tolulope Adesina · NUTM MSc Data Science · 2026

In [ ]:
# ---- Setup: run once per Colab session ----
# 1. Install the course libraries (about two minutes the first time)
!pip install -q statsmodels pmdarima statsforecast neuralforecast lightgbm arch plotly

# 2. Fetch the course data module and data snapshots from the course repository.
#    Replace REPO with your fork or the official course repository URL.
REPO = "https://raw.githubusercontent.com/<your-github-user>/nutdts816/main"
import urllib.request, os
os.makedirs("data", exist_ok=True)
urllib.request.urlretrieve(f"{REPO}/src/tsdata.py", "tsdata.py")
DATA_FILES = ["nigeria_cpi", "nigeria_fx", "nigeria_grid", "nigeria_malaria", "nigeria_rainfall", "bonny_light", "daily_demand",
              "airpassengers", "a10", "h02", "ausbeer", "elecequip", "usmelec", "goog", "nile", "austourists", "oil", "dax", "uschange", "elecdemand"]
for f in DATA_FILES:
    urllib.request.urlretrieve(f"{REPO}/data/{f}.csv", f"data/{f}.csv")

import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (9, 3.6), "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})
import tsdata
print("Setup complete.")

### Carried forward from Lab 1 (run these cells first; they define the objects used below)

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import tsdata   # course data module (real fpp2/R datasets; simulated Nigerian-shaped series)

ap = tsdata.airpassengers()      # monthly international airline passengers, 1949-1960 (thousands)
print(type(ap), ap.index.freq)
print(ap.head(6))
print(ap.index[:3])

In [ ]:
# A messy extract, as you might receive it from a portal
raw = pd.DataFrame({
    'period': ['2024-01', '2024-02', '2024-03', '2024-05', '2024-05', '2024-06', '2024-08'],
    'cpi':    [  743.1,    752.6,    766.0,     787.9,     787.9,    803.4,     828.7]})
raw['date'] = pd.to_datetime(raw['period'], format='%Y-%m')
s = raw.drop_duplicates('date').set_index('date')['cpi']
s = s.asfreq('MS')                       # impose a regular monthly grid; gaps become NaN
print(s)
print('missing:', s.isna().sum())

In [ ]:
s_filled = s.interpolate(method='linear')   # simple, defensible for a smooth monthly index
print(s_filled.round(1))

In [ ]:
dem = tsdata.elecdemand()    # half-hourly electricity demand, Victoria (Australia), 2014
print(dem.head(3))
daily = dem['Demand'].resample('D').sum()          # daily total demand (GW-halfhours)
monthly_mean = dem['Demand'].resample('MS').mean() # monthly mean half-hourly demand
print(daily.head(3).round(1)); print(monthly_mean.round(2).head(3))

In [ ]:
cpi  = tsdata.nigeria_cpi()     # monthly CPI, Jan 2015 - Jun 2026 (simulated)
fx   = tsdata.nigeria_fx()      # monthly NGN/USD, Jan 2015 - Jun 2026 (simulated)
grid = tsdata.nigeria_grid()    # monthly average grid generation, MW (simulated)
print(pd.concat([cpi, fx, grid], axis=1).tail(4))

## Visualisation and autocorrelation

### 1.8 Time plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9, 6))
ap.plot(ax=axes[0,0], title='Airline passengers (thousands), 1949-1960')
cpi.plot(ax=axes[0,1], title='Nigeria CPI, monthly (simulated)')
fx.plot(ax=axes[1,0], title='NGN/USD monthly average (simulated)')
grid.plot(ax=axes[1,1], title='Grid generation, MW (simulated)')
for ax in axes.flat: ax.set_xlabel('')
_caption = 'Four time plots. Read each for trend, seasonality, breaks and changing variability.'

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
ap.plot(ax=axes[0], title='Passengers: original scale')
np.log(ap).plot(ax=axes[1], title='Passengers: log scale (seasonal amplitude now roughly constant)')
for ax in axes: ax.set_xlabel('')
_caption = 'The same series on the original and log scales.'

### 1.9 Seasonal plots and seasonal subseries plots

In [ ]:
def seasonal_plot(s, period_label='year', title=''):
    df = pd.DataFrame({'v': s.values, 'year': s.index.year, 'month': s.index.month})
    piv = df.pivot(index='month', columns='year', values='v')
    ax = piv.plot(legend=False, colormap='viridis', title=title, figsize=(8, 3.4))
    ax.set_xlabel('Month'); ax.set_xticks(range(1, 13))
    for col in piv.columns[-1:]: ax.annotate(str(col), (12, piv[col].iloc[-1]), fontsize=8)
    return ax

seasonal_plot(ap, title='Seasonal plot: airline passengers, one line per year (dark = early, light = late)')
_caption = 'Seasonal plot. The July-August peak and the November trough recur every year; the amplitude grows with the level.'

In [ ]:
seasonal_plot(grid, title='Seasonal plot: grid generation (simulated), one line per year')
_caption = 'The wet-season peak (around August) and dry-season trough (first quarter) are visible but noisier than the airline pattern.'

In [ ]:
def subseries_plot(s, title=''):
    df = pd.DataFrame({'v': s.values, 'year': s.index.year, 'month': s.index.month})
    fig, axes = plt.subplots(1, 12, figsize=(10, 3), sharey=True)
    for m, ax in zip(range(1, 13), axes):
        sub = df[df.month == m]
        ax.plot(sub.year, sub.v, lw=1); ax.axhline(sub.v.mean(), color='#B8860B', lw=1.2)
        ax.set_title(['J','F','M','A','M','J','J','A','S','O','N','D'][m-1]); ax.set_xticks([]); ax.grid(False)
    fig.suptitle(title, y=1.02); return fig

subseries_plot(np.log(ap), 'Seasonal subseries plot: log passengers. Horizontal bar = month mean')
_caption = 'Each panel is one month across the years. The month means trace the seasonal pattern; the within-panel slopes show the trend.'

### 1.10 Lag plots

In [ ]:
def lag_plot_grid(s, lags=(1, 2, 3, 6, 12, 24), title=''):
    fig, axes = plt.subplots(2, 3, figsize=(9, 5.5))
    for k, ax in zip(lags, axes.flat):
        ax.scatter(s.shift(k), s, s=8); ax.set_title(f'lag {k}'); ax.set_xlabel(f'x(t-{k})'); ax.set_ylabel('x(t)')
    fig.suptitle(title, y=1.0); return fig

lag_plot_grid(np.log(ap), title='Lag plots of log airline passengers')
_caption = 'Strong positive linear relationships at every lag (trend), tightest at lags 12 and 24 (annual seasonality).'

In [ ]:
rng = np.random.default_rng(2)
wn = pd.Series(rng.normal(size=144), index=ap.index)
lag_plot_grid(wn, title='Lag plots of white noise')
_caption = 'No structure at any lag: the signature of an unpredictable series.'

### 1.11 Autocorrelation

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
plot_acf(np.log(ap), lags=36, ax=axes[0], title='Log passengers'); 
plot_acf(np.log(ap).diff().dropna(), lags=36, ax=axes[1], title='Log passengers, first difference')
plot_acf(wn, lags=36, ax=axes[2], title='White noise (T=144)')
_caption = 'Three correlograms. Shaded band: the ±1.96/√T bounds.'

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
plot_acf(cpi, lags=36, ax=axes[0], title='CPI level')
plot_acf(np.log(cpi).diff().dropna(), lags=36, ax=axes[1], title='Monthly log change in CPI (inflation)')
plot_acf(np.log(fx).diff().dropna(), lags=36, ax=axes[2], title='Monthly log change in NGN/USD')
_caption = 'The CPI level is dominated by trend; monthly inflation shows strong, slowly decaying positive autocorrelation (persistence: high-inflation months cluster); the exchange-rate change is close to white noise apart from the devaluation jumps.'

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox
for name, s_ in [('white noise', wn), ('log CPI change', np.log(cpi).diff().dropna()), ('log FX change', np.log(fx).diff().dropna())]:
    q = acorr_ljungbox(s_, lags=[12], return_df=True)
    print(f'{name:16s} Q(12) = {q.lb_stat.iloc[0]:8.2f}   p-value = {q.lb_pvalue.iloc[0]:.4f}')

### 1.12 Interactive visualisation

In [ ]:
import plotly.express as px
df = pd.concat([cpi.rename('CPI'), fx.rename('NGN/USD')], axis=1).reset_index().rename(columns={'index': 'date'})
fig = px.line(df, x='date', y=['CPI', 'NGN/USD'], title='CPI and exchange rate (hover, zoom, toggle)')
fig.update_layout(hovermode='x unified')
# fig.show()   # uncomment in Colab

### 1.13 Worked example: first look at a new series

In [ ]:
s = grid
print('Observations:', len(s), '| from', s.index[0].date(), 'to', s.index[-1].date(), '| freq:', s.index.freqstr)
print('Missing:', s.isna().sum(), '| min/median/max:', s.min(), s.median(), s.max())
print('Year means:'); print(s.groupby(s.index.year).mean().round(0).to_string())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 6))
s.plot(ax=axes[0,0], title='Time plot'); s.rolling(12, center=True).mean().plot(ax=axes[0,0], color='#B8860B', label='12-month centred MA'); axes[0,0].legend()
df_ = pd.DataFrame({'v': s.values, 'year': s.index.year, 'month': s.index.month}).pivot(index='month', columns='year', values='v')
df_.plot(ax=axes[0,1], legend=False, colormap='viridis', title='Seasonal plot')
axes[1,0].scatter(s.shift(12), s, s=8); axes[1,0].set_title('Lag-12 plot'); axes[1,0].set_xlabel('x(t-12)')
plot_acf(s, lags=36, ax=axes[1,1], title='ACF')
for ax in axes.flat: ax.set_xlabel(ax.get_xlabel() or '')
_caption = 'A standard four-panel first look.'

## Exercises

4. The sample ACF formula uses $T$ in the denominator rather than $T - k$. Explain in one paragraph why this is a feature and not a bug.
5. Take the simulated `nigeria_cpi()` series and compute the year-on-year inflation rate $100(x_t / x_{t-12} - 1)$. Plot it and its ACF. Is year-on-year inflation more or less autocorrelated than the monthly log change? Why?
6. A colleague plots the ACF of the raw exchange-rate level, sees large autocorrelations at all lags, and concludes that "the exchange rate is highly predictable." Write the two-sentence correction.

In [ ]:
# Your work here
